# 1. 구조화된 티켓 분류

**시나리오:** 자유 형식 고객 문의를 category와 priority 계약으로 변환합니다.

**학습 목표:** Week 1의 입력 검증을 전제로 `TicketClassification` structured output과 분류 실패 범위를 이해합니다.

## 중요 변수·함수

- `TicketRequest`: subject·description·customer tier 입력입니다.
- `TicketClassification`: `billing/access/technical/other`와 `normal/urgent`만 허용합니다.
- `TicketServices.classify`: fixture와 live `with_structured_output()`이 공유하는 경계입니다.

In [ ]:
# 이 학습 Notebook은 외부 API/DB를 사용하지 않는 fixture 모드로 고정합니다.
import os
os.environ["APP_MODE"] = "fixture"

# Notebook 위치에서 실행해도 repository의 canonical app을 가져옵니다.
from pathlib import Path
import sys

_repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# fixture classifier도 live structured output과 같은 Pydantic 결과를 반환합니다.
from week2.app import TicketRequest, create_fixture_services

services = create_fixture_services()
billing = TicketRequest(subject='Duplicate invoice', description='I was charged twice.', customer_tier='standard')
classification = services.classify(billing)
classification.model_dump()

In [ ]:
# 광범위 장애 표현은 category뿐 아니라 priority도 바꿉니다.
outage = TicketRequest(subject='Production outage', description='All users are blocked.', customer_tier='enterprise')
outage_classification = services.classify(outage)
assert outage_classification.category == 'technical'
assert outage_classification.priority == 'urgent'
outage_classification.model_dump()

## 예측 과제와 해석

**예측 과제:** `login` 문의와 `all users blocked` 문의가 각각 어떤 category/priority가 될지 먼저 적으세요.

**해석:** 구조화 출력은 자연어 판단을 graph가 사용할 수 있는 제한된 값으로 바꿉니다. 분류는 실제 환불이나 계정 변경을 수행하지 않습니다.